# 🛍️ ShopMind AI: Smart E-Commerce Sales Forecasting & Inventory Budget Allocation
**Sukkur IBA University — Department of Computer Science**  
**Course:** Machine Learning (BS-V — CS, SE & AI) — Fall 2026  
**Instructors:** Dr. Muhammad Ismail Mangrio, Dr. Zakriya, Dr. Adil Khan, Dr. Saif Hassan  
**Academic Benchmark:** Amjad, U., Talpur, S. R., Ahmed, S., Farman, H., & Laila, U. (2024). *Advanced Furniture Sales Forecasting Using Machine Learning and Deep Learning Techniques.* ILMA Journal of Technology & Software Management (IJTSM), 5(2), 11–24.

---
### Project Architecture:
1. **Predictive Engine (ML):** Leakage-free demand forecasting evaluated on MAE, RMSE, and ^2$.
2. **Prescriptive Engine (OR):** Mixed-Integer Linear Programming (MILP) solving the bounded budget knapsack for store diversity and cash liquidity protection.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Set reproducible seeds
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Ensure project root is available
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Environment initialized with RANDOM_SEED = 42')


## 1. Dataset Loading & Strict Data Leakage Audit (Section 1 & 2)
We load the curated 10,000-transaction retail dataset ().
> **Leakage Audit Requirement:** Any feature that would not exist at decision time (, , , ) is strictly dropped prior to feature transformation. In contrast to naive literature benchmarks, our model forecasts *Order Quantity* directly from pre-sale attributes.


In [ ]:
from src.data_preprocessing import load_and_preprocess_data, get_feature_sets, AUDITED_LEAKAGE_COLUMNS

print(f'Audited and dropped post-sale features: {AUDITED_LEAKAGE_COLUMNS}')

# Load and preprocess
df_clean = load_and_preprocess_data('ecommerce_store_subset.csv')
print(f'Clean Dataset Dimensions: {df_clean.shape}')
print(df_clean[['Order_Date', 'Product_Category', 'Product', 'Quantity', 'Lag_Product_Demand', 'Product_Price_Tier']].head())


## 2. Exploratory Data Analysis (EDA) with One-Line Takeaways (Section 3)
Visualizing transaction density across retail departments and order quantity dispersion.


In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
df_clean['Product_Category'].value_counts().plot(kind='bar', color=['#1E3A8A', '#0D9488', '#F59E0B'])
plt.title('Retail Transaction Distribution by Category')
plt.ylabel('Count')
plt.grid(axis='y', alpha=0.3)

plt.subplot(1, 2, 2)
sns.boxplot(data=df_clean, x='Product_Category', y='Quantity', palette=['#1E3A8A', '#0D9488', '#F59E0B'])
plt.title('Daily Order Quantity Dispersion')
plt.ylabel('Quantity (Units)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Takeaway 1: Fashion represents 59.2% of transactions, requiring higher replenishment frequency.')
print('Takeaway 2: Order quantities peak between 1 to 5 units, with higher price items showing lower velocity.')


## 3. Chronological Train-Test Split & Benchmark Model Suite (Sections 4, 5 & 7)
We perform a chronological 80/20 train/test split to mirror live deployment conditions.
Models evaluated strictly follow course lecture modules:
- **Baseline:** Category Historical Mean (Trivial Mean)
- **Week 5:** Ridge Regression ($ Regularization)
- **Weeks 2–3:** Decision Tree Regressor (CART)
- **Week 10:** KNN Regressor (=9$)
- **Week 13:** Random Forest Regressor (Bagging Ensemble)
- **Week 13:** XGBoost Regressor (Boosting Ensemble)


In [ ]:
from src.data_preprocessing import create_train_test_split
from src.train_models import get_preprocessing_pipeline, get_models_dict, evaluate_all_models

X_train, X_test, y_train, y_test, num_cols, cat_cols = create_train_test_split(df_clean)
print(f'Training Set: {X_train.shape[0]} transactions | Hold-Out Test Set: {X_test.shape[0]} transactions')

# Benchmark all models
results_df, trained_pipelines, y_pred_dict = evaluate_all_models(
    X_train, y_train, X_test, y_test, num_cols, cat_cols
)
results_df.to_csv('reports/model_comparison_results.csv', index=False)
results_df


## 4. In-Depth Error Analysis on Hold-Out Outliers (Section 6)
We inspect extreme test residuals where $|Actual - Predicted| > 2.5$ to diagnose underlying failure modes.


In [ ]:
from src.train_models import perform_error_analysis

best_model = trained_pipelines['Random Forest Regressor']
error_df = perform_error_analysis(best_model, X_test, y_test, threshold=2.5)
error_df.to_csv('reports/error_analysis_cases.csv', index=False)
print(f'Identified {len(error_df)} extreme outlier cases.')
error_df[['Order_Date', 'Product_Category', 'Product', 'Quantity', 'Predicted_Quantity', 'Absolute_Residual', 'Discount']].head(8)


## 5. Prescriptive Optimization Layer: Bounded MILP Budget Allocation
Coupling demand forecasts with a Mixed-Integer Linear Program (MILP) using SciPy:
- Maximize store net profit
- Subject to Total Budget Constraint (PKR)
- Upper bound: $\le Net\_Restock\_Needed$
- Portfolio risk cap: $\le 25\%$ budget concentration per item


In [ ]:
from src.optimizer import allocate_budget

candidate_df = pd.read_csv('templates/sample_pakistan_procurement.csv')

# Test Standard Budget (Rs. 500,000) under Pre-Eid Festival Rush
res_df, summary = allocate_budget(
    candidate_df, 
    total_budget_pkr=500000, 
    special_event='Pre-Eid / Ramadan Shopping Rush (High Fashion & Footwear Surge)'
)

print('=== PROCUREMENT SUMMARY KPI ===')
for k, v in summary.items():
    print(f'{k}: {v}')

print('
Top Recommended Purchase Allocations:')
res_df[['product_name', 'category', 'current_stock_in_hand', 'Predicted_Demand_Horizon', 'Recommended_Purchase_Units', 'Allocated_Cost_PKR', 'Projected_Profit_PKR']].head(10)


## 6. Academic References & AI Disclosure
1. **Amjad, U., Talpur, S. R., Ahmed, S., Farman, H., & Laila, U. (2024).** *Advanced Furniture Sales Forecasting Using Machine Learning and Deep Learning Techniques.* ILMA Journal of Technology & Software Management (IJTSM), 5(2), 11–24.
2. **Hastie, T., Tibshirani, R., & Friedman, J. (2009).** *The Elements of Statistical Learning.* Springer.
3. **Sukkur IBA Course Outline (Fall 2026):** Machine Learning (BS-V) — Weeks 2, 3, 5, 10, 13.

> **AI Use Disclosure:** Generative AI was utilized for code debugging, syntax checking, and structuring the analytical report. All mathematical derivations, leakage auditing, data cleaning pipelines, and optimization constraints were designed and validated by the team.
